In [1]:
# Purpose: find out where Bronze actually lives
from pathlib import Path
for p in Path(".").rglob("bronze"):
    print(p.resolve())


In [2]:
import sys
sys.path.insert(0, "/workspace/src")

import io
import zipfile
import ast
import yaml
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory, postgres_connection
from quantum_lake_student.models import stable_record_hash, QualityFinding, Severity, StageResult
from quantum_lake_student.formats import b8_record_bytes, iter_b8_records, parse_01_records, check_padding_zero
from quantum_lake_student.ml import syndrome_data_split, google_data_split
from quantum_lake_student.io_utils import get_object_bytes, write_parquet, read_parquet, data_issue_row, write_data_issues

settings = Settings.from_environment()
client = minio_client(settings)
bucket = settings.s3_bucket

def get_bronze_bytes(client, bucket, object_name):
    return get_object_bytes(client, bucket, object_name)

bronze_object_syndromes = "bronze/source=qec_syndromes/syndromes_dataset.zip"
bronze_object_google = "bronze/source=google_qec/google-surface-code-curated.zip"
bronze_object_qasmbench = "bronze/source=qasmbench/qasmbench-qec.zip"

syndromes_zip = get_bronze_bytes(client, bucket, bronze_object_syndromes)
google_zip = get_bronze_bytes(client, bucket, bronze_object_google)
qasmbench_zip = get_bronze_bytes(client, bucket, bronze_object_qasmbench)

print("restored: settings, client, bucket, and all three Bronze zip byte blobs")


restored: settings, client, bucket, and all three Bronze zip byte blobs


In [3]:
# Purpose: restore the syndromes parsing function and the 68-record
# prototype from the smallest fault-rate file.
def parse_syndrome_row(row, *, bronze_object, archive_member, csv_row_index, experiment_id, physical_fault_rate):
    rounds = ast.literal_eval(row["syndromes"])
    if len(rounds) != 4 or any(len(r) != 4 for r in rounds):
        return None, {"rule_id": "syndrome_shape", "reason": f"expected 4x4, got shape {[len(r) for r in rounds]}"}
    flat_bits = [v for r in rounds for v in r]
    if any(v not in (0, 1) for v in flat_bits):
        return None, {"rule_id": "syndrome_binary_domain", "reason": "non-binary value in syndrome"}
    quantity = int(row["quantity"])
    if quantity <= 0:
        return None, {"rule_id": "positive_quantity", "reason": f"quantity={quantity}"}
    source_record_id = stable_record_hash(
        {"bronze_object": bronze_object, "archive_member": archive_member, "row": csv_row_index}
    )
    record = {
        "source_record_id": source_record_id,
        "experiment_id": experiment_id,
        "physical_fault_rate": physical_fault_rate,
        "syndrome_bits": bytes(flat_bits),
        "round_count": len(rounds),
        "check_count": len(rounds[0]),
        "logical_error_label": bool(row["labels"]),
        "quantity": quantity,
    }
    return record, None

records = []
with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    member = "d-3_pfr-0.000010_nb-10M.csv"
    with z.open(member) as f:
        df_small = pd.read_csv(f)

for i, row in df_small.iterrows():
    record, issue = parse_syndrome_row(
        row, bronze_object=bronze_object_syndromes, archive_member=member,
        csv_row_index=i, experiment_id="qec_syndromes/d-3_pfr-0.000010", physical_fault_rate=0.00001,
    )
    if record:
        records.append(record)

print("restored records:", len(records))


restored records: 68


In [4]:
# Purpose: restore the Google experiment + shot parsing functions and the
# prototype records (5 experiments, 10 shots of one experiment).
def parse_experiment(z, exp_dir, *, bronze_object):
    props = yaml.safe_load(z.read(f"{exp_dir}/properties.yml"))
    source_record_id = stable_record_hash({"bronze_object": bronze_object, "archive_member": f"{exp_dir}/properties.yml"})
    return {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "basis": props["basis"],
        "distance": props["distance"],
        "rounds": props["rounds"],
        "shots": props["shots"],
        "center_row": props["center_data_qubit_row"],
        "center_col": props["center_data_qubit_col"],
        "measurement_count": props["circuit_measurements"],
        "detector_count": props["circuit_detectors"],
    }

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    exp_dirs = sorted({n.split("/")[0] for n in z.namelist() if "/" in n})
    experiment_records = [parse_experiment(z, d, bronze_object=bronze_object_google) for d in exp_dirs]

PREDICTION_FILES = {
    "belief_matching_prediction": "obs_flips_predicted_by_belief_matching.01",
    "correlated_matching_prediction": "obs_flips_predicted_by_correlated_matching.01",
    "pymatching_prediction": "obs_flips_predicted_by_pymatching.01",
    "tensor_network_contraction_prediction": "obs_flips_predicted_by_tensor_network_contraction.01",
}

def slice_b8_record(data, index, bits_per_record):
    record_bytes = b8_record_bytes(bits_per_record)
    return data[index * record_bytes : (index + 1) * record_bytes]

def parse_shot(files, exp_dir, shot_index, *, experiment_record, bronze_object):
    measurement_bits_packed = slice_b8_record(files["measurements"], shot_index, experiment_record["measurement_count"])
    sweep_bits_packed = slice_b8_record(files["sweep"], shot_index, 9)
    detector_bits_packed = slice_b8_record(files["detectors"], shot_index, experiment_record["detector_count"])
    detector_row = next(iter_b8_records(detector_bits_packed, bits_per_record=experiment_record["detector_count"]))
    detector_event_count = sum(detector_row)
    actual = files["actual"][shot_index]
    predictions = {col: files[col][shot_index] for col in PREDICTION_FILES}
    source_record_id = stable_record_hash({"bronze_object": bronze_object, "experiment_id": exp_dir, "shot_index": shot_index})
    return {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
        "measurement_bits": measurement_bits_packed,
        "sweep_bits": sweep_bits_packed,
        "detector_bits": detector_bits_packed,
        "detector_event_count": detector_event_count,
        "actual_observable_flip": bool(actual),
        **{col: bool(v) for col, v in predictions.items()},
    }

exp_dir = "surface_code_bX_d3_r25_center_3_5"
exp_record = experiment_records[0]

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    files = {
        "measurements": z.read(f"{exp_dir}/measurements.b8"),
        "sweep": z.read(f"{exp_dir}/sweep.b8"),
        "detectors": z.read(f"{exp_dir}/detection_events.b8"),
        "actual": parse_01_records(z.read(f"{exp_dir}/obs_flips_actual.01")),
        **{col: parse_01_records(z.read(f"{exp_dir}/{fname}")) for col, fname in PREDICTION_FILES.items()},
    }

shot_records = [parse_shot(files, exp_dir, i, experiment_record=exp_record, bronze_object=bronze_object_google) for i in range(10)]
print("restored experiment_records:", len(experiment_records), "shot_records:", len(shot_records))


restored experiment_records: 5 shot_records: 10


In [5]:
#Purpose: get_object fetches the zip's bytes from MinIO as a stream;
#io.BytesIO wraps those bytes so zipfile can treat them like a file;
#infolist() lists every member inside the zip without extracting anything to disk.
#This is where you found the 7 CSVs + README.txt.
import io
import zipfile

def get_bronze_bytes(client, bucket, object_name):
    response = client.get_object(bucket, object_name)
    try:
        return response.read()
    finally:
        response.close()
        response.release_conn()

bucket = settings.s3_bucket

syndromes_zip = get_bronze_bytes(
    minio_client(settings), bucket, "bronze/source=qec_syndromes/syndromes_dataset.zip"
)
with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    for info in z.infolist():
        print(info.filename, info.file_size)


d-3_pfr-0.000010_nb-10M.csv 4411
d-3_pfr-0.000050_nb-10M.csv 13715
d-3_pfr-0.000100_nb-10M.csv 31115
d-3_pfr-0.000500_nb-10M.csv 89438
d-3_pfr-0.001000_nb-10M.csv 181222
d-3_pfr-0.005000_nb-10M.csv 1323765
d-3_pfr-0.010000_nb-10M.csv 3148485
README.txt 343


In [6]:
#read README
with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    print(z.read("README.txt").decode())


The file names: d-<surface_code_distance>_pfr-<physical_fault_rate>_nb-<number_of_samples>

The file format is a csv file with the following columns:
- label: binary label (0: no error, 1: error)
- syndromes: syndrome measurement sequence (tuples of the form (round, syndromes))
- quantity: number of samples for this label + syndrome sequence


In [7]:
# Peek the first lines of the samallest CSV file
with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    with z.open("d-3_pfr-0.000010_nb-10M.csv") as f:
        for _ in range(5):
            print(f.readline())


b'labels,syndromes,quantity\n'
b'0,"((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))",9987291\n'
b'0,"((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))",486\n'
b'1,"((0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))",476\n'
b'0,"((0, 1, 0, 0), (0, 1, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))",448\n'


In [8]:
# Load the smallest CSV fully + structural checks
import ast
import pandas as pd

with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    with z.open("d-3_pfr-0.000010_nb-10M.csv") as f:
        df = pd.read_csv(f)

print(df.shape)
print(df.dtypes)

# parse the nested-tuple string into real tuples
df["parsed"] = df["syndromes"].apply(ast.literal_eval)

# check 1: every record has 4 rounds x 4 binary checks
shape_ok = df["parsed"].apply(
    lambda rounds: len(rounds) == 4 and all(len(r) == 4 for r in rounds)
)
print("all rows 4x4:", shape_ok.all())

binary_ok = df["parsed"].apply(
    lambda rounds: all(v in (0, 1) for r in rounds for v in r)
)
print("all values binary:", binary_ok.all())

# check 2: total quantity vs nominal 10M from filename
print("sum of quantity:", df["quantity"].sum())

# check 3: does any syndrome tuple appear with both labels?
dup = df.groupby("syndromes")["labels"].nunique()
print("syndromes with both labels present:", (dup > 1).sum())

# check 4: is (syndromes, labels) unique, or do some rows need a real dedup key?
print("rows:", len(df), "unique (syndromes,labels) pairs:", df.drop_duplicates(subset=["syndromes","labels"]).shape[0])


(68, 3)
labels        int64
syndromes    object
quantity      int64
dtype: object
all rows 4x4: True
all values binary: True
sum of quantity: 10000000
syndromes with both labels present: 0
rows: 68 unique (syndromes,labels) pairs: 68


In [9]:
# Same checks on the largest CSV
with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    with z.open("d-3_pfr-0.010000_nb-10M.csv") as f:
        df_hi = pd.read_csv(f)

df_hi["parsed"] = df_hi["syndromes"].apply(ast.literal_eval)
print(df_hi.shape)
print("sum of quantity:", df_hi["quantity"].sum())

dup_hi = df_hi.groupby("syndromes")["labels"].nunique()
print("syndromes with both labels present:", (dup_hi > 1).sum())
print("rows:", len(df_hi), "unique (syndromes,labels) pairs:", df_hi.drop_duplicates(subset=["syndromes","labels"]).shape[0])


(49676, 4)
sum of quantity: 10000000
syndromes with both labels present: 18210
rows: 49676 unique (syndromes,labels) pairs: 49676


In [10]:
# Purpose: list every archive member inside the qasmbench-qec.zip Bronze object,
# to see the real directory/file structure before parsing anything.
qasmbench_zip = get_bronze_bytes(
    minio_client(settings), bucket, "bronze/source=qasmbench/qasmbench-qec.zip"
)
with zipfile.ZipFile(io.BytesIO(qasmbench_zip)) as z:
    for info in z.infolist():
        print(info.filename, info.file_size)


LICENSE 2066
NOTICE 1285
README.md 27140
qelib1.inc 4090
small/error_correctiond3_n5/README.md 571
small/error_correctiond3_n5/error_correctiond3_n5.png 62789
small/error_correctiond3_n5/error_correctiond3_n5.qasm 1517
small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm 3304
small/error_correctiond3_n5/res_error_correctiond3_n5.png 16042
small/qec_en_n5/README.md 514
small/qec_en_n5/qec_en_n5.png 29104
small/qec_en_n5/qec_en_n5.qasm 530
small/qec_en_n5/qec_en_n5_transpiled.qasm 778
small/qec_en_n5/res_qec_en_n5.png 12183
small/qec_sm_n5/README.md 480
small/qec_sm_n5/qec_sm_n5.png 25706
small/qec_sm_n5/qec_sm_n5.qasm 377
small/qec_sm_n5/qec_sm_n5_transpiled.qasm 341
small/qec_sm_n5/res_qec_sm_n5.png 12085


In [11]:
# Purpose: read the full source QASM and README for qec_sm_n5, the benchmark
# the data-sources guide calls out as the clearest parity-check example.
with zipfile.ZipFile(io.BytesIO(qasmbench_zip)) as z:
    print("--- README ---")
    print(z.read("small/qec_sm_n5/README.md").decode())
    print("--- qec_sm_n5.qasm ---")
    print(z.read("small/qec_sm_n5/qec_sm_n5.qasm").decode())


--- README ---
# Application: qec_sm_n5
- Qubit Count : 5
- Circuit Depth : 10
- Circuit Width : 5
- Retention Lifespan : 2.302585092994046
- Gate Density : 0.14
- Dual Gate Count : 2
- Measurement Density : 1.956011502714073
- Size Factor : 2.5649493574615367
- Gate Count : 5
- Entanglement Variance : 0.4208268308540415
- Communication Supermarq : 0.4
- Measurement Supermarq : 0.0
- Depth Supermarq : 1.0
- Entanglement Supermarq : 0.5
- Parallelism Supermarq : 0
- Liveness Supermarq : 0.3

--- qec_sm_n5.qasm ---
// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



In [12]:
# Purpose: read the full source QASM (not transpiled) for the other two
# benchmarks, to compare their circuit-writing style against qec_sm_n5
# before designing a general stabilizer/correction parser.
with zipfile.ZipFile(io.BytesIO(qasmbench_zip)) as z:
    print("--- qec_en_n5.qasm ---")
    print(z.read("small/qec_en_n5/qec_en_n5.qasm").decode())
    print()
    print("--- error_correctiond3_n5.qasm ---")
    print(z.read("small/error_correctiond3_n5/error_correctiond3_n5.qasm").decode())


--- qec_en_n5.qasm ---
// Name of Experiment: Encoder into bit-flip code with parity checks (qubits 0,1,3) v2

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[2];
t q[2];
h q[2];
h q[0];
h q[1];
h q[2];
cx q[1], q[2];
cx q[0], q[2];
h q[0];
h q[1];
h q[3];
cx q[3], q[2];
h q[2];
h q[3];
cx q[3], q[2];
cx q[0], q[2];
cx q[1], q[2];
h q[2];
h q[4];
cx q[4], q[2];
h q[2];
h q[4];
cx q[4], q[2];
cx q[1], q[2];
cx q[3], q[2];


measure q[2] -> c[2];
measure q[4] -> c[4];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[3] -> c[3];


--- error_correctiond3_n5.qasm ---
// Error correction: distance-three 5-qubit code, from the paper "Benchmarking gate-based quantum computers" by K. Michielsen et al.

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0];
h q[1];
id q[2];
h q[3];
h q[4];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[4],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
sdg q[4];
cx 

In [13]:
# Purpose: read the transpiled QASM for qec_sm_n5 to see whether transpilation
# preserves the custom "syndrome" gate and the if-conditioned corrections,
# or expands/renames them into something a parser would need to handle differently.
with zipfile.ZipFile(io.BytesIO(qasmbench_zip)) as z:
    print(z.read("small/qec_sm_n5/qec_sm_n5_transpiled.qasm").decode())


OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
x q[0];
barrier q[0],q[1],q[2];
cx q[0],a[0];
cx q[1],a[0];
cx q[1],a[1];
cx q[2],a[1];
measure a[0] -> syn[0];
measure a[1] -> syn[1];
if(syn==1) x q[0];
measure q[0] -> c[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q[1] -> c[1];
measure q[2] -> c[2];



In [14]:
# Purpose: list every archive member inside the google-surface-code-curated.zip
# Bronze object, to see the experiment directory structure and companion files
# before parsing anything.
google_zip = get_bronze_bytes(
    minio_client(settings), bucket, "bronze/source=google_qec/google-surface-code-curated.zip"
)
with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    for info in z.infolist():
        print(info.filename, info.file_size)


README.txt 15656
surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem 172277
surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim 18773
surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim 124790
surface_code_bX_d3_r25_center_3_5/detection_events.b8 1250000
surface_code_bX_d3_r25_center_3_5/layout.svg 17685
surface_code_bX_d3_r25_center_3_5/measurements.b8 1350000
surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01 100000
surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_belief_matching.01 100000
surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_correlated_matching.01 100000
surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_pymatching.01 100000
surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_tensor_network_contraction.01 100000
surface_code_bX_d3_r25_center_3_5/pij_from_even_for_odd.dem 121160
surface_code_bX_d3_r25_center_3_5/pij_from_odd_for_even.dem 118405
surface_code_bX_d3_r25_center_3_5/properties.yml 278
surface_code_bX_d3_r25_center

In [15]:
# Purpose: read the full properties.yml for one distance-3 experiment to see
# the exact declared metadata (shots, qubits, measurement/detector counts)
# and confirm it matches the byte-size arithmetic done above.
with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    print(z.read("surface_code_bX_d3_r25_center_3_5/properties.yml").decode())


type: surface_code_memory_experiment
basis: X
rounds: 25
distance: 3
data_qubits: 9
measure_qubits: 8
shots: 50000
center_data_qubit_row: 3
center_data_qubit_col: 5
circuit_measurements: 209
circuit_sweep_bits: 9
circuit_detectors: 200
circuit_observables: 1
circuit_qubits: 17



In [16]:
# Purpose: verify formats.b8_record_bytes() predicts the same byte-per-shot
# counts derived from raw file sizes above, using the declared bit counts
# from properties.yml.
import sys
sys.path.insert(0, "/workspace/src")
from quantum_lake_student.formats import b8_record_bytes

print("measurements:", b8_record_bytes(209), "expected 27")
print("sweep:", b8_record_bytes(9), "expected 2")
print("detectors:", b8_record_bytes(200), "expected 25")


measurements: 27 expected 27
sweep: 2 expected 2
detectors: 25 expected 25


In [17]:
# Purpose: pull one experiment's raw b8/01 bytes from Bronze and decode
# just the first shot of each, to see real values rather than only sizes.
import sys
sys.path.insert(0, "/workspace/src")
from quantum_lake_student.formats import iter_b8_records, parse_01_records

exp_dir = "surface_code_bX_d3_r25_center_3_5"
with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    measurements = z.read(f"{exp_dir}/measurements.b8")
    sweep = z.read(f"{exp_dir}/sweep.b8")
    detectors = z.read(f"{exp_dir}/detection_events.b8")
    actual = z.read(f"{exp_dir}/obs_flips_actual.01")

first_measurement = next(iter_b8_records(measurements, bits_per_record=209))
first_sweep = next(iter_b8_records(sweep, bits_per_record=9))
first_detector = next(iter_b8_records(detectors, bits_per_record=200))
first_actual = parse_01_records(actual)[0]

print("measurement bits (209):", first_measurement)
print("sweep bits (9):", first_sweep)
print("detector bits (200):", first_detector)
print("actual flip (shot 0):", first_actual)


measurement bits (209): (1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1)
sweep bits (9): (0, 0, 0, 0, 1, 0, 1, 1, 0)
detector bits (200): (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [18]:
# Purpose: loop over all 5 experiment directories, parse each properties.yml,
# and verify every companion file's byte/line count matches the declared
# shot count and bit widths (with byte-alignment padding accounted for).
import yaml  # if this import fails, tell me and we'll parse the tiny YAML by hand instead

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    names = sorted({n.split("/")[0] for n in z.namelist() if "/" in n})
    print("experiment dirs:", names)

    for exp in names:
        props = yaml.safe_load(z.read(f"{exp}/properties.yml"))
        shots = props["shots"]

        expected = {
            "measurements.b8": b8_record_bytes(props["circuit_measurements"]) * shots,
            "sweep.b8": b8_record_bytes(props["circuit_sweep_bits"]) * shots,
            "detection_events.b8": b8_record_bytes(props["circuit_detectors"]) * shots,
        }
        actual_sizes = {name: z.getinfo(f"{exp}/{name}").file_size for name in expected}

        print(exp, "distance:", props["distance"], "shots:", shots)
        for name, exp_size in expected.items():
            print(f"  {name}: expected {exp_size}, actual {actual_sizes[name]}, match={exp_size == actual_sizes[name]}")

        for pred_file in [
            "obs_flips_actual.01",
            "obs_flips_predicted_by_belief_matching.01",
            "obs_flips_predicted_by_correlated_matching.01",
            "obs_flips_predicted_by_pymatching.01",
            "obs_flips_predicted_by_tensor_network_contraction.01",
        ]:
            lines = len(z.read(f"{exp}/{pred_file}").splitlines())
            print(f"  {pred_file}: lines={lines}, match={lines == shots}")


experiment dirs: ['surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5']
surface_code_bX_d3_r25_center_3_5 distance: 3 shots: 50000
  measurements.b8: expected 1350000, actual 1350000, match=True
  sweep.b8: expected 100000, actual 100000, match=True
  detection_events.b8: expected 1250000, actual 1250000, match=True
  obs_flips_actual.01: lines=50000, match=True
  obs_flips_predicted_by_belief_matching.01: lines=50000, match=True
  obs_flips_predicted_by_correlated_matching.01: lines=50000, match=True
  obs_flips_predicted_by_pymatching.01: lines=50000, match=True
  obs_flips_predicted_by_tensor_network_contraction.01: lines=50000, match=True
surface_code_bX_d3_r25_center_5_3 distance: 3 shots: 50000
  measurements.b8: expected 1350000, actual 1350000, match=True
  sweep.b8: expected 100000, actual 100000, match=True
  detection_events.b8: expected 1250000,

In [19]:
# Purpose: parse one syndrome CSV row into either a Silver-shaped record
# (matching silver-tables.md's syndrome_observation schema) or a rejection
# reason, applying the checks already prototyped in discovery.
import ast
import sys
sys.path.insert(0, "/workspace/src")
from quantum_lake_student.models import stable_record_hash

def parse_syndrome_row(row, *, bronze_object, archive_member, csv_row_index, experiment_id, physical_fault_rate):
    rounds = ast.literal_eval(row["syndromes"])

    if len(rounds) != 4 or any(len(r) != 4 for r in rounds):
        return None, {"rule_id": "syndrome_shape", "reason": f"expected 4x4, got shape {[len(r) for r in rounds]}"}

    flat_bits = [v for r in rounds for v in r]
    if any(v not in (0, 1) for v in flat_bits):
        return None, {"rule_id": "syndrome_binary_domain", "reason": "non-binary value in syndrome"}

    quantity = int(row["quantity"])
    if quantity <= 0:
        return None, {"rule_id": "positive_quantity", "reason": f"quantity={quantity}"}

    source_record_id = stable_record_hash(
        {"bronze_object": bronze_object, "archive_member": archive_member, "row": csv_row_index}
    )

    record = {
        "source_record_id": source_record_id,
        "experiment_id": experiment_id,
        "physical_fault_rate": physical_fault_rate,
        "syndrome_bits": bytes(flat_bits),   # 16 one-byte binary values
        "round_count": len(rounds),
        "check_count": len(rounds[0]),
        "logical_error_label": bool(row["labels"]),
        "quantity": quantity,
    }
    return record, None


In [20]:
# Purpose: dry-run the parser over all 68 rows of the smallest syndrome file
# and confirm zero rejections, before writing anything to Parquet.
records = []
issues = []

with zipfile.ZipFile(io.BytesIO(syndromes_zip)) as z:
    member = "d-3_pfr-0.000010_nb-10M.csv"
    with z.open(member) as f:
        df_small = pd.read_csv(f)

for i, row in df_small.iterrows():
    record, issue = parse_syndrome_row(
        row,
        bronze_object="bronze/source=qec_syndromes/syndromes_dataset.zip",
        archive_member=member,
        csv_row_index=i,
        experiment_id="qec_syndromes/d-3_pfr-0.000010",
        physical_fault_rate=0.00001,
    )
    if record:
        records.append(record)
    else:
        issues.append(issue)

print("records:", len(records), "issues:", len(issues))
print(records[0])


records: 68 issues: 0
{'source_record_id': 'c9053b81e73e17904c4295088aa409309262ccf229c6bc0e3ad98de8b25f594e', 'experiment_id': 'qec_syndromes/d-3_pfr-0.000010', 'physical_fault_rate': 1e-05, 'syndrome_bits': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'round_count': 4, 'check_count': 4, 'logical_error_label': False, 'quantity': 9987291}


In [21]:
# Purpose: build a PyArrow table from the parsed records with an explicit
# schema matching silver-tables.md's syndrome_observation contract, write it
# to MinIO at the documented Silver path, then read it back to confirm
# it round-trips correctly.
import pyarrow as pa
import pyarrow.parquet as pq

schema = pa.schema([
    ("source_record_id", pa.string()),
    ("experiment_id", pa.string()),
    ("physical_fault_rate", pa.float64()),
    ("syndrome_bits", pa.binary()),
    ("round_count", pa.int32()),
    ("check_count", pa.int32()),
    ("logical_error_label", pa.bool_()),
    ("quantity", pa.int64()),
])

table = pa.Table.from_pylist(records, schema=schema)
print(table.schema)
print("rows:", table.num_rows)

buffer = io.BytesIO()
pq.write_table(table, buffer, compression="zstd")
buffer.seek(0)
payload = buffer.getvalue()

client = minio_client(settings)
target_key = "silver/qec_syndromes/syndrome_observation.parquet"
client.put_object(
    bucket,
    target_key,
    data=io.BytesIO(payload),
    length=len(payload),
    content_type="application/octet-stream",
)
print("wrote", len(payload), "bytes to", target_key)

# read back to confirm
response = client.get_object(bucket, target_key)
try:
    roundtrip_bytes = response.read()
finally:
    response.close()
    response.release_conn()

roundtrip_table = pq.read_table(io.BytesIO(roundtrip_bytes))
print(roundtrip_table.schema)
print(roundtrip_table.slice(0, 3).to_pandas())


source_record_id: string
experiment_id: string
physical_fault_rate: double
syndrome_bits: binary
round_count: int32
check_count: int32
logical_error_label: bool
quantity: int64
rows: 68
wrote 5831 bytes to silver/qec_syndromes/syndrome_observation.parquet
source_record_id: string
experiment_id: string
physical_fault_rate: double
syndrome_bits: binary
round_count: int32
check_count: int32
logical_error_label: bool
quantity: int64
                                    source_record_id  \
0  c9053b81e73e17904c4295088aa409309262ccf229c6bc...   
1  5a0718049e2790acff2e2a356c50f84887bf8836009ea2...   
2  50b56bf8efe6c557b40fa35d9abbcdb2963f0778227149...   

                    experiment_id  physical_fault_rate  \
0  qec_syndromes/d-3_pfr-0.000010              0.00001   
1  qec_syndromes/d-3_pfr-0.000010              0.00001   
2  qec_syndromes/d-3_pfr-0.000010              0.00001   

                                       syndrome_bits  round_count  \
0  b'\x00\x00\x00\x00\x00\x00\x00\x00\x0

In [22]:
# Purpose: same as before, but use a cursor explicitly since executemany
# is a cursor method in psycopg3, not a Connection method.
with postgres_connection(settings) as conn:
    with conn.transaction():
        conn.execute("""
            DROP TABLE IF EXISTS syndrome_observation;
            CREATE TABLE syndrome_observation (
                source_record_id     text PRIMARY KEY,
                experiment_id        text NOT NULL,
                physical_fault_rate  double precision NOT NULL,
                syndrome_bits        bytea NOT NULL,
                round_count          integer NOT NULL CHECK (round_count = 4),
                check_count          integer NOT NULL CHECK (check_count = 4),
                logical_error_label  boolean NOT NULL,
                quantity             bigint NOT NULL CHECK (quantity > 0)
            )
        """)
        with conn.cursor() as cur:
            cur.executemany(
                """
                INSERT INTO syndrome_observation
                    (source_record_id, experiment_id, physical_fault_rate, syndrome_bits,
                     round_count, check_count, logical_error_label, quantity)
                VALUES (%(source_record_id)s, %(experiment_id)s, %(physical_fault_rate)s,
                        %(syndrome_bits)s, %(round_count)s, %(check_count)s,
                        %(logical_error_label)s, %(quantity)s)
                """,
                records,
            )

with postgres_connection(settings) as conn:
    result = conn.execute("SELECT count(*), sum(quantity) FROM syndrome_observation").fetchone()
    print("row count:", result[0], "quantity sum:", result[1])
    sample = conn.execute("SELECT * FROM syndrome_observation LIMIT 3").fetchall()
    for row in sample:
        print(row)


row count: 68 quantity sum: 10000000
('c9053b81e73e17904c4295088aa409309262ccf229c6bc0e3ad98de8b25f594e', 'qec_syndromes/d-3_pfr-0.000010', 1e-05, b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 4, 4, False, 9987291)
('5a0718049e2790acff2e2a356c50f84887bf8836009ea263dfe4bc5b0d30a820', 'qec_syndromes/d-3_pfr-0.000010', 1e-05, b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00', 4, 4, False, 486)
('50b56bf8efe6c557b40fa35d9abbcdb2963f07782271494149d7a2e0853dc403', 'qec_syndromes/d-3_pfr-0.000010', 1e-05, b'\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 4, 4, True, 476)


In [23]:
# Purpose: query the Gold syndrome_observation table, compute the ML contract
# columns (example_id via hash, data_split via the supplied helper), and
# write the result as a real ml_syndrome_decoder_example.parquet slice.
from quantum_lake_student.ml import syndrome_data_split

with postgres_connection(settings) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT experiment_id, physical_fault_rate, syndrome_bits,
                   round_count, check_count, logical_error_label, quantity
            FROM syndrome_observation
        """)
        gold_rows = cur.fetchall()
        columns = [desc.name for desc in cur.description]

ml_records = []
for row in gold_rows:
    values = dict(zip(columns, row))
    syndrome_list = list(values["syndrome_bits"])  # bytes -> list[int] for hashing/schema

    example_id = stable_record_hash({
        "experiment_id": values["experiment_id"],
        "syndrome_bits": syndrome_list,
        "logical_error_label": values["logical_error_label"],
    })

    ml_records.append({
        "example_id": example_id,
        "experiment_id": values["experiment_id"],
        "physical_fault_rate": values["physical_fault_rate"],
        "syndrome_bits": values["syndrome_bits"],  # keep as bytes for the binary column
        "round_count": values["round_count"],
        "check_count": values["check_count"],
        "logical_error_label": values["logical_error_label"],
        "sample_weight": values["quantity"],
        "data_split": syndrome_data_split(values["physical_fault_rate"]),
    })

ml_schema = pa.schema([
    ("example_id", pa.string()),
    ("experiment_id", pa.string()),
    ("physical_fault_rate", pa.float64()),
    ("syndrome_bits", pa.binary()),
    ("round_count", pa.int32()),
    ("check_count", pa.int32()),
    ("logical_error_label", pa.bool_()),
    ("sample_weight", pa.int64()),
    ("data_split", pa.string()),
])

ml_table = pa.Table.from_pylist(ml_records, schema=ml_schema)
print(ml_table.schema)
print("rows:", ml_table.num_rows)
print("unique example_ids:", len(set(ml_table.column("example_id").to_pylist())))
print("splits present:", set(ml_table.column("data_split").to_pylist()))

ml_buffer = io.BytesIO()
pq.write_table(ml_table, ml_buffer, compression="zstd")
ml_payload = ml_buffer.getvalue()

ml_key = "ml/ml_syndrome_decoder_example.parquet"
client.put_object(bucket, ml_key, data=io.BytesIO(ml_payload), length=len(ml_payload), content_type="application/octet-stream")
print("wrote", len(ml_payload), "bytes to", ml_key)


example_id: string
experiment_id: string
physical_fault_rate: double
syndrome_bits: binary
round_count: int32
check_count: int32
logical_error_label: bool
sample_weight: int64
data_split: string
rows: 68
unique example_ids: 68
splits present: {'train'}
wrote 6104 bytes to ml/ml_syndrome_decoder_example.parquet


In [24]:
# Purpose: parse one Google experiment directory's properties.yml into a
# Silver-shaped `experiment` record, and derive experiment_id from the
# directory name so it's stable across reruns.
import yaml

def parse_experiment(z, exp_dir, *, bronze_object):
    props = yaml.safe_load(z.read(f"{exp_dir}/properties.yml"))

    source_record_id = stable_record_hash({
        "bronze_object": bronze_object,
        "archive_member": f"{exp_dir}/properties.yml",
    })

    record = {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,  # directory name already encodes basis/distance/rounds/center
        "basis": props["basis"],
        "distance": props["distance"],
        "rounds": props["rounds"],
        "shots": props["shots"],
        "center_row": props["center_data_qubit_row"],
        "center_col": props["center_data_qubit_col"],
        "measurement_count": props["circuit_measurements"],
        "detector_count": props["circuit_detectors"],
    }
    return record

bronze_object_google = "bronze/source=google_qec/google-surface-code-curated.zip"

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    exp_dirs = sorted({n.split("/")[0] for n in z.namelist() if "/" in n})
    experiment_records = [parse_experiment(z, exp_dir, bronze_object=bronze_object_google) for exp_dir in exp_dirs]

for rec in experiment_records:
    print(rec)


{'source_record_id': '413433113642867c41b42965d89ad41b30b869eb1a78a139b2ab10f9157fed1d', 'experiment_id': 'surface_code_bX_d3_r25_center_3_5', 'basis': 'X', 'distance': 3, 'rounds': 25, 'shots': 50000, 'center_row': 3, 'center_col': 5, 'measurement_count': 209, 'detector_count': 200}
{'source_record_id': 'cf41523c12170afaeef02adb9cd76f88aa44356bdf6c5f78999709aef59d3cda', 'experiment_id': 'surface_code_bX_d3_r25_center_5_3', 'basis': 'X', 'distance': 3, 'rounds': 25, 'shots': 50000, 'center_row': 5, 'center_col': 3, 'measurement_count': 209, 'detector_count': 200}
{'source_record_id': 'a89e384ca3d4273e0df48c5005635739a49e451f17900e51d52ee5280755fbec', 'experiment_id': 'surface_code_bX_d3_r25_center_5_7', 'basis': 'X', 'distance': 3, 'rounds': 25, 'shots': 50000, 'center_row': 5, 'center_col': 7, 'measurement_count': 209, 'detector_count': 200}
{'source_record_id': '1b83a7e3dd78459aef987cc5998c38661bab0729dbcf559c09e2e8bb6eddc693', 'experiment_id': 'surface_code_bX_d3_r25_center_7_5', 'b

In [25]:
# Purpose: build one Silver `shot` record for shot 0 of one experiment,
# pulling measurements/sweep/detectors plus the actual + 4 predicted flip
# files, and cross-checking detector_event_count against the real bits.
from quantum_lake_student.formats import iter_b8_records, parse_01_records

PREDICTION_FILES = {
    "belief_matching_prediction": "obs_flips_predicted_by_belief_matching.01",
    "correlated_matching_prediction": "obs_flips_predicted_by_correlated_matching.01",
    "pymatching_prediction": "obs_flips_predicted_by_pymatching.01",
    "tensor_network_contraction_prediction": "obs_flips_predicted_by_tensor_network_contraction.01",
}

def parse_shot(z, exp_dir, shot_index, *, experiment_record, bronze_object):
    measurement_bits_per_record = experiment_record["measurement_count"]
    detector_bits_per_record = experiment_record["detector_count"]

    measurements = z.read(f"{exp_dir}/measurements.b8")
    sweep = z.read(f"{exp_dir}/sweep.b8")
    detectors = z.read(f"{exp_dir}/detection_events.b8")

    measurement_row = list(iter_b8_records(measurements, bits_per_record=measurement_bits_per_record))[shot_index]
    sweep_row = list(iter_b8_records(sweep, bits_per_record=9))[shot_index]  # sweep bit width from properties.yml, hardcoded here for the prototype
    detector_row = list(iter_b8_records(detectors, bits_per_record=detector_bits_per_record))[shot_index]

    detector_event_count = sum(detector_row)

    actual = parse_01_records(z.read(f"{exp_dir}/obs_flips_actual.01"))[shot_index]
    predictions = {
        column: parse_01_records(z.read(f"{exp_dir}/{fname}"))[shot_index]
        for column, fname in PREDICTION_FILES.items()
    }

    source_record_id = stable_record_hash({
        "bronze_object": bronze_object,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
    })

    measurement_bytes = bytes(
        sum(bit << (i % 8) for i, bit in enumerate(measurement_row[byte * 8:(byte + 1) * 8]))
        for byte in range((len(measurement_row) + 7) // 8)
    )
    # NOTE: repacking bits back to bytes here just to store the packed form in Silver;
    # we'll sanity check this repacking matches the original bytes before trusting it.

    record = {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
        "detector_event_count": detector_event_count,
        "actual_observable_flip": bool(actual),
        **{col: bool(v) for col, v in predictions.items()},
    }
    return record, detector_row

exp_dir = "surface_code_bX_d3_r25_center_3_5"
exp_record = experiment_records[0]

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    shot_record, detector_row = parse_shot(
        z, exp_dir, 0, experiment_record=exp_record, bronze_object=bronze_object_google
    )

print(shot_record)
print("detector bits sum check:", sum(detector_row), "==", shot_record["detector_event_count"])


{'source_record_id': 'cdb13538a4fb6bcb43decdaeaa11999f55eeb77847fb18a8fadefc50efc49902', 'experiment_id': 'surface_code_bX_d3_r25_center_3_5', 'shot_index': 0, 'detector_event_count': 27, 'actual_observable_flip': True, 'belief_matching_prediction': True, 'correlated_matching_prediction': True, 'pymatching_prediction': True, 'tensor_network_contraction_prediction': True}
detector bits sum check: 27 == 27


In [26]:
# Purpose: get the packed measurement/sweep/detector bytes for one shot by
# slicing the raw b8 file directly at record boundaries -- simpler and safer
# than unpacking bits and repacking them, since it never risks a bit-order bug.
from quantum_lake_student.formats import b8_record_bytes

def slice_b8_record(data: bytes, index: int, bits_per_record: int) -> bytes:
    record_bytes = b8_record_bytes(bits_per_record)
    return data[index * record_bytes : (index + 1) * record_bytes]

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    measurements = z.read(f"{exp_dir}/measurements.b8")
    sweep = z.read(f"{exp_dir}/sweep.b8")
    detectors = z.read(f"{exp_dir}/detection_events.b8")

measurement_bits_packed = slice_b8_record(measurements, 0, exp_record["measurement_count"])
sweep_bits_packed = slice_b8_record(sweep, 0, 9)
detector_bits_packed = slice_b8_record(detectors, 0, exp_record["detector_count"])

print("measurement bytes:", measurement_bits_packed, len(measurement_bits_packed))
print("sweep bytes:", sweep_bits_packed, len(sweep_bits_packed))
print("detector bytes:", detector_bits_packed, len(detector_bits_packed))

# cross-check: unpacking this raw slice should give the exact same bits as before
from quantum_lake_student.formats import iter_b8_records
recheck = next(iter_b8_records(detector_bits_packed, bits_per_record=exp_record["detector_count"]))
print("re-decoded detector bits match earlier decode:", list(recheck) == list(detector_row))


measurement bytes: b'\x85 \xad\x905\x90_\xb8\x1d\xb8\x1d\xb8]\xf8]\xf8_\xf8]\xd8\x07\x80\x05\xa2\x0c\x08\x01' 27
sweep bytes: b'\xd0\x00' 2
detector bytes: b'\x00\x80\x82\t\x00\xa0&\x04\x00\x00\x00\x00\x04\x00\x00  \x00\x00\xa2\'\x02"\xb0\x00' 25
re-decoded detector bits match earlier decode: True


In [27]:
# Purpose: final version of parse_shot, using direct byte-slicing for the
# packed fields (measurement_bits/sweep_bits/detector_bits) instead of
# unpack+repack. Run over the first 10 shots of one experiment as our
# small end-to-end batch, matching the syndromes prototype's scale.
from quantum_lake_student.formats import iter_b8_records, parse_01_records, b8_record_bytes

def slice_b8_record(data: bytes, index: int, bits_per_record: int) -> bytes:
    record_bytes = b8_record_bytes(bits_per_record)
    return data[index * record_bytes : (index + 1) * record_bytes]

def parse_shot(files, exp_dir, shot_index, *, experiment_record, bronze_object):
    measurement_bits_packed = slice_b8_record(files["measurements"], shot_index, experiment_record["measurement_count"])
    sweep_bits_packed = slice_b8_record(files["sweep"], shot_index, 9)
    detector_bits_packed = slice_b8_record(files["detectors"], shot_index, experiment_record["detector_count"])

    detector_row = next(iter_b8_records(detector_bits_packed, bits_per_record=experiment_record["detector_count"]))
    detector_event_count = sum(detector_row)

    actual = files["actual"][shot_index]
    predictions = {col: files[col][shot_index] for col in PREDICTION_FILES}

    source_record_id = stable_record_hash({
        "bronze_object": bronze_object,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
    })

    return {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
        "measurement_bits": measurement_bits_packed,
        "sweep_bits": sweep_bits_packed,
        "detector_bits": detector_bits_packed,
        "detector_event_count": detector_event_count,
        "actual_observable_flip": bool(actual),
        **{col: bool(v) for col, v in predictions.items()},
    }

exp_dir = "surface_code_bX_d3_r25_center_3_5"
exp_record = experiment_records[0]

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    files = {
        "measurements": z.read(f"{exp_dir}/measurements.b8"),
        "sweep": z.read(f"{exp_dir}/sweep.b8"),
        "detectors": z.read(f"{exp_dir}/detection_events.b8"),
        "actual": parse_01_records(z.read(f"{exp_dir}/obs_flips_actual.01")),
        **{col: parse_01_records(z.read(f"{exp_dir}/{fname}")) for col, fname in PREDICTION_FILES.items()},
    }

shot_records = [
    parse_shot(files, exp_dir, i, experiment_record=exp_record, bronze_object=bronze_object_google)
    for i in range(10)
]

for rec in shot_records:
    print(rec["shot_index"], rec["detector_event_count"], rec["actual_observable_flip"],
          [rec[col] for col in PREDICTION_FILES])


0 27 True [True, True, True, True]
1 36 True [True, False, False, True]
2 32 True [True, True, False, True]
3 25 False [False, True, True, False]
4 52 True [False, False, False, True]
5 28 False [True, True, True, False]
6 16 False [True, False, False, False]
7 34 True [True, True, False, True]
8 48 False [False, False, False, True]
9 38 True [False, True, True, True]


In [28]:
# Purpose: build PyArrow tables for the Google `experiment` and `shot`
# Silver tables with explicit schemas matching silver-tables.md exactly,
# write both to MinIO, and read them back to confirm.
experiment_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("experiment_id", pa.string()),
    ("basis", pa.string()),
    ("distance", pa.int32()),
    ("rounds", pa.int32()),
    ("shots", pa.int64()),
    ("center_row", pa.int32()),
    ("center_col", pa.int32()),
    ("measurement_count", pa.int32()),
    ("detector_count", pa.int32()),
])

shot_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("experiment_id", pa.string()),
    ("shot_index", pa.int64()),
    ("measurement_bits", pa.binary()),
    ("sweep_bits", pa.binary()),
    ("detector_bits", pa.binary()),
    ("detector_event_count", pa.int32()),
    ("actual_observable_flip", pa.bool_()),
    ("belief_matching_prediction", pa.bool_()),
    ("correlated_matching_prediction", pa.bool_()),
    ("pymatching_prediction", pa.bool_()),
    ("tensor_network_contraction_prediction", pa.bool_()),
])

def write_parquet_to_minio(client, bucket, key, table):
    buffer = io.BytesIO()
    pq.write_table(table, buffer, compression="zstd")
    payload = buffer.getvalue()
    client.put_object(bucket, key, data=io.BytesIO(payload), length=len(payload), content_type="application/octet-stream")
    return len(payload)

experiment_table = pa.Table.from_pylist(experiment_records, schema=experiment_schema)
shot_table = pa.Table.from_pylist(shot_records, schema=shot_schema)

exp_bytes = write_parquet_to_minio(client, bucket, "silver/google_qec/experiment.parquet", experiment_table)
shot_bytes = write_parquet_to_minio(client, bucket, "silver/google_qec/shot.parquet", shot_table)
print("wrote experiment.parquet:", exp_bytes, "bytes,", experiment_table.num_rows, "rows")
print("wrote shot.parquet:", shot_bytes, "bytes,", shot_table.num_rows, "rows")

# read back to confirm
for key in ["silver/google_qec/experiment.parquet", "silver/google_qec/shot.parquet"]:
    response = client.get_object(bucket, key)
    try:
        data = response.read()
    finally:
        response.close()
        response.release_conn()
    t = pq.read_table(io.BytesIO(data))
    print(key, "->", t.schema.names, t.num_rows, "rows")


wrote experiment.parquet: 3611 bytes, 5 rows
wrote shot.parquet: 5001 bytes, 10 rows
silver/google_qec/experiment.parquet -> ['source_record_id', 'experiment_id', 'basis', 'distance', 'rounds', 'shots', 'center_row', 'center_col', 'measurement_count', 'detector_count'] 5 rows
silver/google_qec/shot.parquet -> ['source_record_id', 'experiment_id', 'shot_index', 'measurement_bits', 'sweep_bits', 'detector_bits', 'detector_event_count', 'actual_observable_flip', 'belief_matching_prediction', 'correlated_matching_prediction', 'pymatching_prediction', 'tensor_network_contraction_prediction'] 10 rows


In [29]:
# Purpose: verify the unused padding bits at the end of each packed record
# are genuinely zero, not just assumed -- a real required check per the brief
# ("Stim b8 length, byte alignment, bit order, and unused padding bits").
def check_padding_zero(record_bytes: bytes, bits_per_record: int) -> bool:
    used_bits_in_last_byte = bits_per_record % 8
    if used_bits_in_last_byte == 0:
        return True  # no padding at all, e.g. detector_bits at 200/600 bits
    mask = (0xFF << used_bits_in_last_byte) & 0xFF
    return (record_bytes[-1] & mask) == 0

for rec in shot_records:
    measurement_ok = check_padding_zero(rec["measurement_bits"], exp_record["measurement_count"])
    sweep_ok = check_padding_zero(rec["sweep_bits"], 9)
    print(rec["shot_index"], "measurement padding zero:", measurement_ok, "sweep padding zero:", sweep_ok)


0 measurement padding zero: True sweep padding zero: True
1 measurement padding zero: True sweep padding zero: True
2 measurement padding zero: True sweep padding zero: True
3 measurement padding zero: True sweep padding zero: True
4 measurement padding zero: True sweep padding zero: True
5 measurement padding zero: True sweep padding zero: True
6 measurement padding zero: True sweep padding zero: True
7 measurement padding zero: True sweep padding zero: True
8 measurement padding zero: True sweep padding zero: True
9 measurement padding zero: True sweep padding zero: True


In [30]:
# Purpose: verify the newly added formats.check_padding_zero against real
# Google shot data -- restart the kernel first (or re-run the import cell)
# so the updated formats.py is picked up.
from quantum_lake_student.formats import check_padding_zero

for rec in shot_records:
    print(
        rec["shot_index"],
        "measurement:", check_padding_zero(rec["measurement_bits"], exp_record["measurement_count"]),
        "sweep:", check_padding_zero(rec["sweep_bits"], 9),
    )


0 measurement: True sweep: True
1 measurement: True sweep: True
2 measurement: True sweep: True
3 measurement: True sweep: True
4 measurement: True sweep: True
5 measurement: True sweep: True
6 measurement: True sweep: True
7 measurement: True sweep: True
8 measurement: True sweep: True
9 measurement: True sweep: True


In [31]:
# Purpose: sanity-check that predictions are correctly aligned to actual
# outcomes by measuring decoder agreement rate over a larger sample -- a
# misalignment bug would show up as ~50% agreement (random), not
# decoder-like performance.
sample_size = 2000
sample_shots = [
    parse_shot(files, exp_dir, i, experiment_record=exp_record, bronze_object=bronze_object_google)
    for i in range(sample_size)
]

for col in PREDICTION_FILES:
    matches = sum(1 for s in sample_shots if s[col] == s["actual_observable_flip"])
    print(col, "agreement rate:", matches / sample_size)


belief_matching_prediction agreement rate: 0.592
correlated_matching_prediction agreement rate: 0.553
pymatching_prediction agreement rate: 0.557
tensor_network_contraction_prediction agreement rate: 0.584


In [32]:
# Purpose: diagnose whether the ~55-60% decoder agreement is a real result
# or a sign of misalignment, by checking (1) class balance of actual flips,
# (2) whether decoders agree with EACH OTHER (they should, even if not with
# actual, if they're all reading the same correctly-aligned detector data),
# and (3) a majority-baseline comparison.
actual_flips = [s["actual_observable_flip"] for s in sample_shots]
print("actual flip class balance: True =", sum(actual_flips) / sample_size, "False =", 1 - sum(actual_flips) / sample_size)

# majority baseline: always predict the more common actual value
majority_value = sum(actual_flips) / sample_size > 0.5
majority_agreement = sum(1 for f in actual_flips if f == majority_value) / sample_size
print("majority-baseline agreement:", majority_agreement)

# pairwise agreement between decoders themselves
decoder_cols = list(PREDICTION_FILES)
for i in range(len(decoder_cols)):
    for j in range(i + 1, len(decoder_cols)):
        col_a, col_b = decoder_cols[i], decoder_cols[j]
        agree = sum(1 for s in sample_shots if s[col_a] == s[col_b]) / sample_size
        print(col_a, "vs", col_b, "agreement:", agree)


actual flip class balance: True = 0.489 False = 0.511
majority-baseline agreement: 0.511
belief_matching_prediction vs correlated_matching_prediction agreement: 0.702
belief_matching_prediction vs pymatching_prediction agreement: 0.675
belief_matching_prediction vs tensor_network_contraction_prediction agreement: 0.735
correlated_matching_prediction vs pymatching_prediction agreement: 0.718
correlated_matching_prediction vs tensor_network_contraction_prediction agreement: 0.619
pymatching_prediction vs tensor_network_contraction_prediction agreement: 0.604


In [33]:
# Purpose: verify io_utils.data_issue_row + write_data_issues against a real
# QualityFinding instance, writing to local disk (current signature).
import importlib
import quantum_lake_student.io_utils as io_utils_module
importlib.reload(io_utils_module)
from quantum_lake_student.io_utils import data_issue_row, write_data_issues, read_local_parquet
from quantum_lake_student.models import QualityFinding, Severity

sample_finding = QualityFinding(
    rule_id="syndrome_shape",
    severity=Severity.ERROR,
    source_system="qec_syndromes",
    source_record_locator="d-3_pfr-0.000010_nb-10M.csv:row=5",
    message="expected 4x4 syndrome shape, got malformed row",
    observed_value="((0,0),(0,0,0,0),(0,0,0,0),(0,0,0,0))",
)

row = data_issue_row(sample_finding, run_id="test-run-1", action="rejected", source_record_id="abc123")
print(row)

local_path = "/workspace/results/part1/data_issues_test.parquet"
n = write_data_issues(local_path, [row])
print("wrote", n, "bytes to", local_path)

readback = read_local_parquet(local_path)
print(readback.schema)
print(readback.to_pandas())


{'issue_id': '0ffe910522c2d42e42709ed9bad1340bc7bf651d85bc983357cecaa992c88d64', 'run_id': 'test-run-1', 'source_record_id': 'abc123', 'rule_id': 'syndrome_shape', 'severity': 'error', 'observed_value': '((0,0),(0,0,0,0),(0,0,0,0),(0,0,0,0))', 'action': 'rejected', 'reason': 'expected 4x4 syndrome shape, got malformed row'}
wrote 2984 bytes to /workspace/results/part1/data_issues_test.parquet
issue_id: string
run_id: string
source_record_id: string
rule_id: string
severity: string
observed_value: string
action: string
reason: string
                                            issue_id      run_id  \
0  0ffe910522c2d42e42709ed9bad1340bc7bf651d85bc98...  test-run-1   

  source_record_id         rule_id severity  \
0           abc123  syndrome_shape    error   

                          observed_value    action  \
0  ((0,0),(0,0,0,0),(0,0,0,0),(0,0,0,0))  rejected   

                                           reason  
0  expected 4x4 syndrome shape, got malformed row  


In [34]:
# Purpose: verify the revised write_data_issues writes to local disk (inside
# the workspace container, which maps to starter/results/ on your host) and
# reads back correctly, now that results/ is confirmed as a local-fs area.
import importlib
import quantum_lake_student.io_utils as io_utils_module
importlib.reload(io_utils_module)
from quantum_lake_student.io_utils import data_issue_row, write_data_issues, read_local_parquet

row = data_issue_row(sample_finding, run_id="test-run-1", action="rejected", source_record_id="abc123")

local_path = "/workspace/results/part1/data_issues_test.parquet"
n = write_data_issues(local_path, [row])
print("wrote", n, "bytes to", local_path)

readback = read_local_parquet(local_path)
print(readback.schema)
print(readback.to_pandas())


wrote 2984 bytes to /workspace/results/part1/data_issues_test.parquet
issue_id: string
run_id: string
source_record_id: string
rule_id: string
severity: string
observed_value: string
action: string
reason: string
                                            issue_id      run_id  \
0  0ffe910522c2d42e42709ed9bad1340bc7bf651d85bc98...  test-run-1   

  source_record_id         rule_id severity  \
0           abc123  syndrome_shape    error   

                          observed_value    action  \
0  ((0,0),(0,0,0,0),(0,0,0,0),(0,0,0,0))  rejected   

                                           reason  
0  expected 4x4 syndrome shape, got malformed row  


In [35]:
# Purpose: estimate storage cost of two Gold detector-storage designs --
# (A) one row per shot with packed bytes + event count (what we've built),
# vs (B) one row per individual FIRED detector event -- using real average
# event counts from our sample, to make an evidence-based design decision.
import statistics

# use the 2000-shot sample from earlier if still in memory; otherwise fall back to the 10-shot one
sample = sample_shots if "sample_shots" in dir() else shot_records
avg_events_d3 = statistics.mean(s["detector_event_count"] for s in sample)
print("average detector_event_count per shot (sample, distance 3):", avg_events_d3)

total_shots = 250_000  # 5 experiments x 50,000 shots
d3_shots = 4 * 50_000
d5_shots = 1 * 50_000

# Design A: shot-level summary (what Silver already stores)
# one row per shot: fixed columns + packed bytes (~25 bytes d3, ~75 bytes d5 for detector_bits alone)
bytes_per_shot_row_d3 = 100  # rough: ids, counts, booleans, plus ~25B detector_bits + ~27B measurement_bits + ~2B sweep_bits
bytes_per_shot_row_d5 = 250  # same shape, larger packed fields (~75B detectors + ~79B measurements + ~4B sweep)
design_a_bytes = d3_shots * bytes_per_shot_row_d3 + d5_shots * bytes_per_shot_row_d5
print("Design A (shot summary) estimated size:", design_a_bytes / 1e6, "MB")

# Design B: one row per fired detector event
# assume similar average event rate holds roughly across d3 experiments (real number would need
# per-experiment measurement, this is a bounding estimate not a precise count)
bytes_per_event_row = 60  # experiment_id + shot_index + detector_position + indexes, roughly
design_b_rows_d3 = d3_shots * avg_events_d3
# distance 5 has 3x the detector width (600 vs 200); assume event rate scales roughly with width as a rough bound
design_b_rows_d5 = d5_shots * avg_events_d3 * 3
design_b_rows = design_b_rows_d3 + design_b_rows_d5
design_b_bytes = design_b_rows * bytes_per_event_row
print("Design B (per-fired-event) estimated rows:", design_b_rows, "estimated size:", design_b_bytes / 1e6, "MB")

print("Design B is approximately", round(design_b_bytes / design_a_bytes, 1), "x larger than Design A")


average detector_event_count per shot (sample, distance 3): 31.08
Design A (shot summary) estimated size: 32.5 MB
Design B (per-fired-event) estimated rows: 10878000.0 estimated size: 652.68 MB
Design B is approximately 20.1 x larger than Design A


In [36]:
# Purpose: fix parse_shot to take the sweep-bit width as a parameter
# (read from properties.yml per experiment) instead of hardcoding 9,
# which was silently wrong for any non-distance-3 experiment.
def parse_shot(files, exp_dir, shot_index, *, experiment_record, sweep_bits_count, bronze_object):
    measurement_bits_packed = slice_b8_record(files["measurements"], shot_index, experiment_record["measurement_count"])
    sweep_bits_packed = slice_b8_record(files["sweep"], shot_index, sweep_bits_count)
    detector_bits_packed = slice_b8_record(files["detectors"], shot_index, experiment_record["detector_count"])

    detector_row = next(iter_b8_records(detector_bits_packed, bits_per_record=experiment_record["detector_count"]))
    detector_event_count = sum(detector_row)

    actual = files["actual"][shot_index]
    predictions = {col: files[col][shot_index] for col in PREDICTION_FILES}

    source_record_id = stable_record_hash({"bronze_object": bronze_object, "experiment_id": exp_dir, "shot_index": shot_index})

    return {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
        "measurement_bits": measurement_bits_packed,
        "sweep_bits": sweep_bits_packed,
        "detector_bits": detector_bits_packed,
        "detector_event_count": detector_event_count,
        "actual_observable_flip": bool(actual),
        **{col: bool(v) for col, v in predictions.items()},
    }


In [37]:
# Purpose: check the distance-5 experiment's real average detector_event_count
# against the distance-3-derived estimate used in the storage decision, using
# the corrected parse_shot with the real per-experiment sweep-bit width.
exp_dir_d5 = "surface_code_bX_d5_r25_center_5_5"
exp_record_d5 = next(r for r in experiment_records if r["experiment_id"] == exp_dir_d5)

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    props_d5 = yaml.safe_load(z.read(f"{exp_dir_d5}/properties.yml"))
    sweep_bits_count_d5 = props_d5["circuit_sweep_bits"]

    files_d5 = {
        "measurements": z.read(f"{exp_dir_d5}/measurements.b8"),
        "sweep": z.read(f"{exp_dir_d5}/sweep.b8"),
        "detectors": z.read(f"{exp_dir_d5}/detection_events.b8"),
        "actual": parse_01_records(z.read(f"{exp_dir_d5}/obs_flips_actual.01")),
        **{col: parse_01_records(z.read(f"{exp_dir_d5}/{fname}")) for col, fname in PREDICTION_FILES.items()},
    }

sample_size_d5 = 2000
sample_d5 = [
    parse_shot(files_d5, exp_dir_d5, i, experiment_record=exp_record_d5, sweep_bits_count=sweep_bits_count_d5, bronze_object=bronze_object_google)
    for i in range(sample_size_d5)
]

avg_events_d5 = statistics.mean(s["detector_event_count"] for s in sample_d5)
print("distance-5 sweep bit width:", sweep_bits_count_d5)
print("average detector_event_count per shot (sample, distance 5):", avg_events_d5)
print("previous extrapolated estimate was:", avg_events_d3 * 3)


distance-5 sweep bit width: 25
average detector_event_count per shot (sample, distance 5): 95.1215
previous extrapolated estimate was: 93.24


In [38]:
# Purpose: scale up Google shot extraction across all 5 experiments, 5,000
# shots each (25,000 total, vs. the earlier 10-shot/1-experiment prototype),
# using the corrected parse_shot (real per-experiment sweep-bit width, no
# more hardcoded 9), and reconcile counts.
import time

SAMPLE_PER_EXPERIMENT = 5000
all_google_shots = []

start = time.time()
with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    for exp_record in experiment_records:
        exp_dir = exp_record["experiment_id"]
        props = yaml.safe_load(z.read(f"{exp_dir}/properties.yml"))
        sweep_bits_count = props["circuit_sweep_bits"]

        files = {
            "measurements": z.read(f"{exp_dir}/measurements.b8"),
            "sweep": z.read(f"{exp_dir}/sweep.b8"),
            "detectors": z.read(f"{exp_dir}/detection_events.b8"),
            "actual": parse_01_records(z.read(f"{exp_dir}/obs_flips_actual.01")),
            **{col: parse_01_records(z.read(f"{exp_dir}/{fname}")) for col, fname in PREDICTION_FILES.items()},
        }

        n = min(SAMPLE_PER_EXPERIMENT, exp_record["shots"])
        exp_shots = [
            parse_shot(files, exp_dir, i, experiment_record=exp_record, sweep_bits_count=sweep_bits_count, bronze_object=bronze_object_google)
            for i in range(n)
        ]
        all_google_shots.extend(exp_shots)
        print(exp_dir, "distance:", exp_record["distance"], "parsed:", len(exp_shots))

elapsed = time.time() - start
print()
print("TOTAL shots parsed:", len(all_google_shots), "in", round(elapsed, 1), "seconds")
print("unique source_record_ids:", len({s["source_record_id"] for s in all_google_shots}))


surface_code_bX_d3_r25_center_3_5 distance: 3 parsed: 5000
surface_code_bX_d3_r25_center_5_3 distance: 3 parsed: 5000
surface_code_bX_d3_r25_center_5_7 distance: 3 parsed: 5000
surface_code_bX_d3_r25_center_7_5 distance: 3 parsed: 5000
surface_code_bX_d5_r25_center_5_5 distance: 5 parsed: 5000

TOTAL shots parsed: 25000 in 0.8 seconds
unique source_record_ids: 25000


In [39]:
# Purpose: validate padding-zero across the full 25,000-shot batch, then
# write the scaled-up Silver tables to MinIO, overwriting the earlier
# 10-row prototype with this much more complete slice.
padding_failures = [
    s["source_record_id"] for s in all_google_shots
    if not check_padding_zero(s["measurement_bits"], next(
        r["measurement_count"] for r in experiment_records if r["experiment_id"] == s["experiment_id"]
    ))
]
print("measurement padding failures:", len(padding_failures))

shot_table_full = pa.Table.from_pylist(all_google_shots, schema=shot_schema)
print("shot_table_full rows:", shot_table_full.num_rows)

exp_bytes = write_parquet_to_minio(client, bucket, "silver/google_qec/experiment.parquet", experiment_table)
shot_bytes = write_parquet_to_minio(client, bucket, "silver/google_qec/shot.parquet", shot_table_full)
print("rewrote experiment.parquet:", exp_bytes, "bytes")
print("rewrote shot.parquet:", shot_bytes, "bytes,", shot_table_full.num_rows, "rows")

readback = pq.read_table(io.BytesIO(get_object_bytes(client, bucket, "silver/google_qec/shot.parquet")))
print("read back rows:", readback.num_rows)
print(readback.to_pandas()["experiment_id"].value_counts())


measurement padding failures: 0
shot_table_full rows: 25000
rewrote experiment.parquet: 3611 bytes
rewrote shot.parquet: 2554465 bytes, 25000 rows
read back rows: 25000
experiment_id
surface_code_bX_d3_r25_center_3_5    5000
surface_code_bX_d3_r25_center_5_3    5000
surface_code_bX_d3_r25_center_5_7    5000
surface_code_bX_d3_r25_center_7_5    5000
surface_code_bX_d5_r25_center_5_5    5000
Name: count, dtype: int64


In [40]:
# Purpose: check sweep_bits padding-zero across all 25,000 shots (the
# field we just fixed the hardcoded-width bug for), which was skipped in
# the earlier full-batch check.
sweep_padding_failures = []
for s in all_google_shots:
    exp_id = s["experiment_id"]
    props_lookup = next(r for r in experiment_records if r["experiment_id"] == exp_id)
    # sweep width isn't in experiment_records (not a Silver column); look it up per experiment once
    sweep_padding_failures.append((s["source_record_id"], s["sweep_bits"]))

# faster: precompute sweep width per experiment once, then check
with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    sweep_widths = {
        r["experiment_id"]: yaml.safe_load(z.read(f"{r['experiment_id']}/properties.yml"))["circuit_sweep_bits"]
        for r in experiment_records
    }

failures = [
    s["source_record_id"] for s in all_google_shots
    if not check_padding_zero(s["sweep_bits"], sweep_widths[s["experiment_id"]])
]
print("sweep padding failures:", len(failures))


sweep padding failures: 0


In [41]:
# Purpose: verify check_safe_archive_member against real archive member
# names from all three sources (should all pass) and against deliberately
# unsafe examples (should all fail), to confirm the check actually works.
import importlib
import quantum_lake_student.io_utils as io_utils_module
importlib.reload(io_utils_module)
from quantum_lake_student.io_utils import check_safe_archive_member

all_real_members = []
for zip_bytes in [syndromes_zip, google_zip, qasmbench_zip]:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        all_real_members.extend(z.namelist())

unsafe_real = [m for m in all_real_members if not check_safe_archive_member(m)]
print("real members checked:", len(all_real_members), "flagged unsafe:", len(unsafe_real))
print(unsafe_real)

unsafe_examples = ["../../etc/passwd", "/etc/passwd", "C:/Windows/system32", "a/../../b", "normal/path/file.csv"]
for name in unsafe_examples:
    print(name, "-> safe:", check_safe_archive_member(name))


real members checked: 103 flagged unsafe: 0
[]
../../etc/passwd -> safe: False
/etc/passwd -> safe: False
C:/Windows/system32 -> safe: False
a/../../b -> safe: False
normal/path/file.csv -> safe: True


In [42]:
# Purpose: verify the generalized check_required_members works for both
# google_qec (per-experiment companion files) and qasmbench (per-benchmark
# source+transpiled QASM), proving the function is genuinely source-agnostic.
import importlib
import quantum_lake_student.io_utils as io_utils_module
importlib.reload(io_utils_module)
from quantum_lake_student.io_utils import check_required_members

REQUIRED_GOOGLE_FILES = [
    "properties.yml", "measurements.b8", "sweep.b8", "detection_events.b8",
    "obs_flips_actual.01",
    "obs_flips_predicted_by_belief_matching.01",
    "obs_flips_predicted_by_correlated_matching.01",
    "obs_flips_predicted_by_pymatching.01",
    "obs_flips_predicted_by_tensor_network_contraction.01",
]

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    google_available = set(z.namelist())

for exp_record in experiment_records:
    missing = check_required_members(google_available, exp_record["experiment_id"], REQUIRED_GOOGLE_FILES)
    print(exp_record["experiment_id"], "missing:", missing)

# qasmbench: each benchmark needs its source + transpiled QASM
with zipfile.ZipFile(io.BytesIO(qasmbench_zip)) as z:
    qasm_available = set(z.namelist())

for benchmark in ["error_correctiond3_n5", "qec_en_n5", "qec_sm_n5"]:
    required = [f"{benchmark}.qasm", f"{benchmark}_transpiled.qasm"]
    missing = check_required_members(qasm_available, f"small/{benchmark}", required)
    print(benchmark, "missing:", missing)


surface_code_bX_d3_r25_center_3_5 missing: []
surface_code_bX_d3_r25_center_5_3 missing: []
surface_code_bX_d3_r25_center_5_7 missing: []
surface_code_bX_d3_r25_center_7_5 missing: []
surface_code_bX_d5_r25_center_5_5 missing: []
error_correctiond3_n5 missing: []
qec_en_n5 missing: []
qec_sm_n5 missing: []


In [43]:
# Purpose: give parse_shot a real rejection path, folding the padding-zero
# check in directly instead of running it separately after the fact, and
# prove it actually rejects a corrupted case (not just always succeeding).
def parse_shot(files, exp_dir, shot_index, *, experiment_record, sweep_bits_count, bronze_object):
    measurement_bits_packed = slice_b8_record(files["measurements"], shot_index, experiment_record["measurement_count"])
    sweep_bits_packed = slice_b8_record(files["sweep"], shot_index, sweep_bits_count)
    detector_bits_packed = slice_b8_record(files["detectors"], shot_index, experiment_record["detector_count"])

    source_record_id = stable_record_hash({"bronze_object": bronze_object, "experiment_id": exp_dir, "shot_index": shot_index})
    locator = f"{exp_dir}:shot={shot_index}"

    if not check_padding_zero(measurement_bits_packed, experiment_record["measurement_count"]):
        return None, QualityFinding(
            rule_id="measurement_padding_nonzero", severity=Severity.ERROR, source_system="google_qec",
            source_record_locator=locator, message="unused padding bits in measurement_bits are not zero",
            observed_value=measurement_bits_packed.hex(),
        )
    if not check_padding_zero(sweep_bits_packed, sweep_bits_count):
        return None, QualityFinding(
            rule_id="sweep_padding_nonzero", severity=Severity.ERROR, source_system="google_qec",
            source_record_locator=locator, message="unused padding bits in sweep_bits are not zero",
            observed_value=sweep_bits_packed.hex(),
        )

    detector_row = next(iter_b8_records(detector_bits_packed, bits_per_record=experiment_record["detector_count"]))
    detector_event_count = sum(detector_row)
    actual = files["actual"][shot_index]
    predictions = {col: files[col][shot_index] for col in PREDICTION_FILES}

    record = {
        "source_record_id": source_record_id,
        "experiment_id": exp_dir,
        "shot_index": shot_index,
        "measurement_bits": measurement_bits_packed,
        "sweep_bits": sweep_bits_packed,
        "detector_bits": detector_bits_packed,
        "detector_event_count": detector_event_count,
        "actual_observable_flip": bool(actual),
        **{col: bool(v) for col, v in predictions.items()},
    }
    return record, None

# prove the rejection path actually works: corrupt a real record's padding bit and confirm it's caught
good_record, good_issue = parse_shot(files, exp_dir, 0, experiment_record=exp_record, sweep_bits_count=9, bronze_object=bronze_object_google)
print("normal shot -> record:", good_record is not None, "issue:", good_issue)

# Purpose: corrupt one real measurement record's padding bits directly,
# then confirm parse_shot's new rejection path actually catches it.
corrupted_measurements = bytearray(files["measurements"])
corrupted_measurements[26] |= 0b11000000  # force some padding bits (positions 6-7 of byte 26) to 1
corrupted_files = dict(files)
corrupted_files["measurements"] = bytes(corrupted_measurements)

bad_record, bad_issue = parse_shot(corrupted_files, exp_dir, 0, experiment_record=exp_record, sweep_bits_count=9, bronze_object=bronze_object_google)
print("corrupted shot -> record:", bad_record, "issue:", bad_issue)



normal shot -> record: False issue: QualityFinding(rule_id='sweep_padding_nonzero', severity=<Severity.ERROR: 'error'>, source_system='google_qec', source_record_locator='surface_code_bX_d5_r25_center_5_5:shot=0', message='unused padding bits in sweep_bits are not zero', observed_value='d036')
corrupted shot -> record: None issue: QualityFinding(rule_id='sweep_padding_nonzero', severity=<Severity.ERROR: 'error'>, source_system='google_qec', source_record_locator='surface_code_bX_d5_r25_center_5_5:shot=0', message='unused padding bits in sweep_bits are not zero', observed_value='d036')


In [44]:
 # Purpose: re-run the good/corrupted parse_shot test with explicitly
# fetched distance-3 data, instead of relying on exp_dir/exp_record/files
# left over from the earlier scale-up loop's last iteration (which was
# distance-5, causing the previous mismatched-width result).
exp_dir_d3 = "surface_code_bX_d3_r25_center_3_5"
exp_record_d3 = next(r for r in experiment_records if r["experiment_id"] == exp_dir_d3)

with zipfile.ZipFile(io.BytesIO(google_zip)) as z:
    files_d3 = {
        "measurements": z.read(f"{exp_dir_d3}/measurements.b8"),
        "sweep": z.read(f"{exp_dir_d3}/sweep.b8"),
        "detectors": z.read(f"{exp_dir_d3}/detection_events.b8"),
        "actual": parse_01_records(z.read(f"{exp_dir_d3}/obs_flips_actual.01")),
        **{col: parse_01_records(z.read(f"{exp_dir_d3}/{fname}")) for col, fname in PREDICTION_FILES.items()},
    }

good_record, good_issue = parse_shot(files_d3, exp_dir_d3, 0, experiment_record=exp_record_d3, sweep_bits_count=9, bronze_object=bronze_object_google)
print("normal shot -> record:", good_record is not None, "issue:", good_issue)

corrupted_measurements = bytearray(files_d3["measurements"])
corrupted_measurements[26] |= 0b11000000
corrupted_files_d3 = dict(files_d3)
corrupted_files_d3["measurements"] = bytes(corrupted_measurements)

bad_record, bad_issue = parse_shot(corrupted_files_d3, exp_dir_d3, 0, experiment_record=exp_record_d3, sweep_bits_count=9, bronze_object=bronze_object_google)
print("corrupted shot -> record:", bad_record, "issue:", bad_issue)


normal shot -> record: True issue: None
corrupted shot -> record: None issue: QualityFinding(rule_id='measurement_padding_nonzero', severity=<Severity.ERROR: 'error'>, source_system='google_qec', source_record_locator='surface_code_bX_d3_r25_center_3_5:shot=0', message='unused padding bits in measurement_bits are not zero', observed_value='8520ad9035905fb81db81db85df85df85ff85dd8078005a20c08c1')


In [45]:
# Purpose: verify register_sources.run() against the real running platform --
# should succeed cleanly (all 3 checksums match, no unsafe members, no
# unexpected objects), since nothing about Bronze has changed.
import importlib
import quantum_lake_student.stages.register_sources as register_sources_module
importlib.reload(register_sources_module)
from quantum_lake_student.stages.register_sources import run as register_sources_run

result = register_sources_run("test-run-register")
print(result)


StageResult(stage='register_sources', run_id='test-run-register', input_count=3, output_count=3, issue_count=0, started_at=datetime.datetime(2026, 9, 22, 17, 18, 27, 405980, tzinfo=datetime.timezone.utc), finished_at=datetime.datetime(2026, 9, 22, 17, 18, 27, 436838, tzinfo=datetime.timezone.utc))


In [47]:
# Purpose: run the complete merged prepare_data.py dispatcher for real,
# writing all three sources' Silver tables to MinIO and results/part1/ to
# local disk -- the one integration path not yet tested (host can't reach
# /workspace).
import importlib
import quantum_lake_student.stages.prepare_data as prepare_data_module
importlib.reload(prepare_data_module)

result = prepare_data_module.run("final-check-run")
print(result)


StageResult(stage='prepare_data', run_id='final-check-run', input_count=325604, output_count=325619, issue_count=24, started_at=datetime.datetime(2026, 9, 22, 17, 19, 7, 230544, tzinfo=datetime.timezone.utc), finished_at=datetime.datetime(2026, 9, 22, 17, 19, 26, 514485, tzinfo=datetime.timezone.utc))
